In [6]:
# ============================================================
# MODELO FINAL XGBOOST MEJORADO PARA CHURN
# Optimiza churn, no accuracy
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier


# ============================================================
# 1. CARGAR DATASET
# ============================================================

df_model = pd.read_csv("dataset_limpio_churn.csv")

# Copia del dataset
df = df_model.copy()

print("=" * 60)
print("SHAPE INICIAL")
print("=" * 60)
print(df.shape)

print("\nDistribución de churn:")
print(df["churn"].value_counts())
print(df["churn"].value_counts(normalize=True) * 100)


# ============================================================
# 2. QUITAR COLUMNAS QUE NO DEBEN ENTRAR
# ============================================================

columnas_eliminar = [
    "cliente_id",
    "fecha",
    "zona_id",
    "zona_id_x",
    "zona_id_y"
]

for col in columnas_eliminar:
    if col in df.columns:
        df = df.drop(columns=col)


# ============================================================
# 3. TARGET
# ============================================================

target = "churn"

X = df.drop(columns=[target])
y = df[target]


# ============================================================
# 4. VARIABLES CATEGÓRICAS
# ============================================================

X = pd.get_dummies(X, drop_first=True)


# ============================================================
# 5. LIMPIEZA DE NULOS
# ============================================================

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())


# ============================================================
# 6. TRAIN / TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 7. PESO PARA CLASE MINORITARIA
# ============================================================

negativos = (y_train == 0).sum()
positivos = (y_train == 1).sum()

scale_pos_weight = negativos / positivos

print("="*60)
print("DISTRIBUCIÓN TRAIN")
print("="*60)
print(y_train.value_counts())
print(y_train.value_counts(normalize=True) * 100)

print("\nscale_pos_weight:", round(scale_pos_weight, 2))


# ============================================================
# 8. MODELO XGBOOST
# ============================================================

modelo = XGBClassifier(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    gamma=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1,
    reg_lambda=5,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train, y_train)


# ============================================================
# 9. PROBABILIDADES
# ============================================================

proba = modelo.predict_proba(X_test)[:, 1]


# ============================================================
# 10. BUSCAR THRESHOLD MANUALMENTE
# Queremos evitar accuracy artificial de 98%
# ============================================================

resultados = []

for threshold in np.arange(0.10, 0.81, 0.01):

    pred_temp = (proba >= threshold).astype(int)

    acc = accuracy_score(y_test, pred_temp)
    prec = precision_score(y_test, pred_temp, zero_division=0)
    rec = recall_score(y_test, pred_temp, zero_division=0)
    f1 = f1_score(y_test, pred_temp, zero_division=0)

    resultados.append({
        "threshold": threshold,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    })

df_thresholds = pd.DataFrame(resultados)


# ============================================================
# 11. ELEGIR THRESHOLD DEFENDIBLE
# Objetivo: recall mínimo 0.25 y mejor F1 posible
# ============================================================

candidatos = df_thresholds[
    (df_thresholds["recall"] >= 0.25) &
    (df_thresholds["accuracy"] <= 0.95)
]

if len(candidatos) > 0:
    mejor = candidatos.sort_values("f1", ascending=False).iloc[0]
else:
    mejor = df_thresholds.sort_values("f1", ascending=False).iloc[0]

threshold_final = mejor["threshold"]

pred = (proba >= threshold_final).astype(int)


# ============================================================
# 12. MÉTRICAS FINALES
# ============================================================

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)
roc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)


print("\n" + "="*70)
print("RESULTADOS MODELO FINAL XGBOOST MEJORADO")
print("="*70)

print("Threshold utilizado:", round(threshold_final, 2))
print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision churn:", round(precision, 4))
print("Recall churn:", round(recall, 4))
print("F1 churn:", round(f1, 4))
print("ROC-AUC:", round(roc, 4))
print("PR-AUC:", round(pr_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred, zero_division=0))


# ============================================================
# 13. TABLA DE THRESHOLDS PARA JUSTIFICAR
# ============================================================

print("\n" + "="*70)
print("COMPARACIÓN DE THRESHOLDS")
print("="*70)

print(
    df_thresholds[
        (df_thresholds["threshold"] >= 0.10) &
        (df_thresholds["threshold"] <= 0.80)
    ][["threshold", "accuracy", "precision", "recall", "f1"]]
    .sort_values("f1", ascending=False)
    .head(15)
)


# ============================================================
# 14. IMPORTANCIA DE VARIABLES
# ============================================================

importancias = pd.DataFrame({
    "Variable": X.columns,
    "Importancia": modelo.feature_importances_
}).sort_values("Importancia", ascending=False)

print("\n" + "="*70)
print("TOP 20 VARIABLES MÁS IMPORTANTES")
print("="*70)
print(importancias.head(20))

SHAPE INICIAL
(317579, 35)

Distribución de churn:
churn
0    315611
1      1968
Name: count, dtype: int64
churn
0    99.380312
1     0.619688
Name: proportion, dtype: float64
DISTRIBUCIÓN TRAIN
churn
0    252489
1      1574
Name: count, dtype: int64
churn
0    99.380469
1     0.619531
Name: proportion, dtype: float64

scale_pos_weight: 160.41

RESULTADOS MODELO FINAL XGBOOST MEJORADO
Threshold utilizado: 0.62
Accuracy: 91.1 %
Precision churn: 0.0181
Recall churn: 0.2513
F1 churn: 0.0339
ROC-AUC: 0.689
PR-AUC: 0.0171

Matriz de confusión:
[[57766  5356]
 [  295    99]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.92      0.95     63122
           1       0.02      0.25      0.03       394

    accuracy                           0.91     63516
   macro avg       0.51      0.58      0.49     63516
weighted avg       0.99      0.91      0.95     63516


COMPARACIÓN DE THRESHOLDS
    threshold  accuracy  precision    recall  

El modelo XGBoost permitió abordar el fuerte desbalanceo existente en los datos mediante el uso del parámetro `scale_pos_weight`, sin necesidad de aplicar técnicas de sobremuestreo. El modelo alcanzó una exactitud global del 91,10 % y un valor de ROC-AUC de 0,6890, lo que indica una capacidad discriminativa moderada. En cuanto a la detección de clientes con riesgo de abandono, consiguió identificar aproximadamente el 25,13 % de los casos reales de churn (recall = 0,2513), aunque con una precisión reducida (0,0181) debido al elevado número de falsos positivos. A pesar de ello, el modelo fue capaz de detectar una parte significativa de los clientes con riesgo de abandono en un problema altamente desbalanceado. Sin embargo, los resultados obtenidos fueron inferiores a los alcanzados por otras configuraciones evaluadas, especialmente en términos de recall, F1-score y capacidad discriminativa, por lo que este modelo no fue finalmente seleccionado como modelo definitivo.
